In [ ]:
#  INSTALL LIBRARIES & AUTHENTICATE
!pip -q install --upgrade google-cloud-bigquery

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import google.cloud.bigquery as bq, os, uuid, json, pandas as pd

In [ ]:
PROJECT_ID   = "qwiklabs-gcp-02-e659e41ff1eb"
DATASET_ID   = "fraud_demo"
RAW_TABLE_ID = "fraud_data_raw"
FEAT_TABLE_ID= "fraud_training_data"
GCS_URI      = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

client = bq.Client(project=PROJECT_ID)
dataset_ref = client.dataset(DATASET_ID)
raw_table   = dataset_ref.table(RAW_TABLE_ID)
feat_table  = dataset_ref.table(FEAT_TABLE_ID)

In [ ]:
#CREATE DATASET
try:
    client.get_dataset(dataset_ref)
    print(f"Dataset `{DATASET_ID}` already exists.")
except Exception:
    ds = bq.Dataset(dataset_ref)
    ds.location = "US"
    client.create_dataset(ds)
    print(f"Dataset `{DATASET_ID}` created.")

Dataset `fraud_demo` already exists.


In [ ]:
#LOAD THE RAW CSV INTO fraud_data_raw
job_cfg = bq.LoadJobConfig(
    source_format       = bq.SourceFormat.CSV,
    skip_leading_rows   = 1,
    autodetect          = True,
    write_disposition   = "WRITE_TRUNCATE",
)

load_job = client.load_table_from_uri(GCS_URI, raw_table, job_config=job_cfg)
print("Starting load job:", load_job.job_id)
load_job.result()
print("Finished: rows =", client.get_table(raw_table).num_rows)

Starting load job: ad1019fa-5731-43b4-981f-861dfd96b14f
Finished: rows = 50000


In [ ]:
# FEATURE ENGINEERING & ONE-HOT ENCODING
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{FEAT_TABLE_ID}` AS

WITH base AS (
  SELECT
    *,
    SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,
    DATE_DIFF(Application_Date,
              Previous_Assistance_Date,
              DAY) AS Time_Since_Previous_Assistance
  FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE_ID}`
),

encoded AS (
  SELECT
    -- keep everything except the two categoricals & raw Age
    * EXCEPT(Employment_Status, Device_Type, Age),

    /* ───── One-hot encode Employment_Status ───── */
    CAST(Employment_Status = 'Employed'   AS INT64) AS Employment_Status_Employed,
    CAST(Employment_Status = 'Unemployed' AS INT64) AS Employment_Status_Unemployed,
    CAST(Employment_Status = 'Student'    AS INT64) AS Employment_Status_Student,
    CAST(Employment_Status = 'Retired'    AS INT64) AS Employment_Status_Retired,

    /* ───── One-hot encode Device_Type ───── */
    CAST(Device_Type = 'Mobile'  AS INT64) AS Device_Type_Mobile,
    CAST(Device_Type = 'Desktop' AS INT64) AS Device_Type_Desktop,
    CAST(Device_Type = 'Tablet'  AS INT64) AS Device_Type_Tablet,

    /* ───── Age bins + one-hot ───── */
    CAST(Age BETWEEN 18 AND 24 AS INT64) AS Age_18_24,
    CAST(Age BETWEEN 25 AND 34 AS INT64) AS Age_25_34,
    CAST(Age BETWEEN 35 AND 44 AS INT64) AS Age_35_44,
    CAST(Age BETWEEN 45 AND 54 AS INT64) AS Age_45_54,
    CAST(Age >= 55            AS INT64) AS Age_55_plus,

    /* ───── Example: convert a BOOL column to 0/1 ───── */
    CAST(FRAUDULENT AS INT64) AS Is_Fraud
  FROM base
)

SELECT *
FROM encoded;
"""



In [ ]:
# Run the script
job = client.query(sql)
job.result()                      # waits for completion
print(f"Created `{DATASET_ID}.{FEAT_TABLE_ID}` ✓")

Created `fraud_demo.fraud_training_data` ✓
